### Which products have the highest revenue?

In [2]:
from dotenv import load_dotenv
from databricks.connect import DatabricksSession
import os

load_dotenv() 

spark = DatabricksSession.builder \
    .host(os.getenv("DATABRICKS_HOST")) \
    .token(os.getenv("DATABRICKS_TOKEN")) \
    .serverless(True) \
    .getOrCreate()

In [3]:
spark.sql("""
SELECT
product_id,
product_name,
SUM(revenue) AS product_revenue,
SUM(quantity) AS units_sold
FROM gold.fct_order_items
GROUP BY product_id,product_name
ORDER BY product_revenue DESC
LIMIT 20;
""").toPandas()

,product_id,product_name,product_revenue,units_sold
0,prod_003,USB-C Hub,23634.00,303
1,prod_006,Monitor Stand,21289.50,171
2,prod_005,Webcam HD,21271.00,239
3,prod_016,Running Shoes,18330.00,195
4,prod_002,Bluetooth Speaker,15931.00,178
5,prod_004,Mechanical Keyboard,15209.43,181
6,prod_011,Desk Lamp,11830.26,174
7,prod_018,Sunglasses,11055.00,201
8,prod_012,Yoga Mat,9090.00,202
9,prod_009,Online Course Pack,8698.26,174


### Which categories drive the most revenue?

In [0]:
spark.sql("""
SELECT
    product_category AS category,
    SUM(revenue)     AS category_revenue,
    SUM(quantity)    AS units_sold
FROM gold.fct_order_items
GROUP BY product_category
ORDER BY category_revenue DESC;
""").toPandas()


### What is average order value (AOV) by month?

In [4]:
spark.sql("""
SELECT
    DATE_TRUNC('month', order_date)        AS month,
    SUM(revenue)                           AS total_revenue,
    COUNT(DISTINCT order_id)               AS order_count,
    SUM(revenue) / COUNT(DISTINCT order_id) AS aov
FROM gold.fct_orders
GROUP BY 1
ORDER BY 1;
""").toPandas()


,month,total_revenue,order_count,aov
0,2024-01-01,9244.15,24,385.172916666666666667
1,2024-02-01,18953.47,52,364.489807692307692308
2,2024-03-01,18627.05,37,503.433783783783783784
3,2024-04-01,15067.13,40,376.678250000000000000
4,2024-05-01,22680.21,49,462.861428571428571429
5,2024-06-01,14432.01,36,400.889166666666666667
6,2024-07-01,17525.65,44,398.310227272727272727
7,2024-08-01,23307.35,56,416.202678571428571429
8,2024-09-01,9540.89,27,353.366296296296296296
9,2024-10-01,20561.44,47,437.477446808510638298


###  Which countries generate the most revenue?

In [5]:
spark.sql("""
SELECT
    customer_country            AS country,
    SUM(revenue)                AS revenue,
    COUNT(DISTINCT order_id)    AS order_count
FROM gold.fct_orders
GROUP BY customer_country
ORDER BY revenue DESC
LIMIT 10;
""").toPandas()


,country,revenue,order_count
0,UK,29425.55,65
1,AU,28021.79,63
2,JP,27239.70,61
3,US,26613.93,71
4,FR,25812.71,64
5,SG,23063.17,58
6,DE,23017.31,52
7,CA,22294.46,66


### Who are the top customers by revenue?

In [7]:
spark.sql("""
SELECT
    o.customer_id,
    MAX(c.email)        AS email,
    MAX(c.country)      AS country,
    SUM(o.revenue)      AS total_revenue,
    COUNT(o.order_id)   AS order_count
FROM gold.fct_orders o
LEFT JOIN gold.dim_customers c
    ON o.customer_id = c.customer_id
GROUP BY o.customer_id
ORDER BY total_revenue DESC
LIMIT 20;
""").toPandas()


,customer_id,email,country,total_revenue,order_count
0,cust_0765,uri.davis204@email.com,JP,2672.95,3
1,cust_0157,diana.powell822@email.com,FR,2574.52,4
2,cust_0680,victor.baker79@email.com,AU,2178.89,3
3,cust_0310,frank.turner979@email.com,UK,2144.15,3
4,cust_0052,ryan.wallace134@email.com,AU,2140.92,4
5,cust_0380,maya.coleman677@email.com,SG,2105.45,3
6,cust_0651,kevin.sullivan949@email.com,JP,1947.76,4
7,cust_0195,leo.hamilton544@email.com,DE,1906.44,2
8,cust_0736,paula.gonzales523@email.com,JP,1714.05,3
9,cust_0285,xia.reed528@email.com,UK,1696.40,3


### Who are the customers who have more than one order (repeat customers)?

In [8]:
spark.sql("""
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.no_of_orders
FROM (
    SELECT
        customer_id,
        COUNT(order_id) as no_of_orders
    FROM workspace.gold.fct_orders
    WHERE customer_id IS NOT NULL
      AND order_id IS NOT NULL
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
) o
JOIN workspace.gold.dim_customers c
    ON o.customer_id = c.customer_id;
""").toPandas()


,customer_id,first_name,last_name,no_of_orders
0,cust_0020,Holly,Johnson,2
1,cust_0030,Ryan,Anderson,2
2,cust_0037,Bella,Murphy,2
3,cust_0042,Zack,Perry,2
4,cust_0052,Ryan,Wallace,4
...,...,...,...,...
89,cust_0948,Zara,Henderson,2
90,cust_0970,Piper,Howard,2
91,cust_0971,Eve,Perry,2
92,cust_0988,Chris,Jenkins,3
